# Reordering mesh geometrical entities using Morton or Hilbert curves


## Introduction
 
**Authors**: Adrien Bruneton, Pierre Ledac

**Created**: 06/2025

## Description
 
This validation form explains how the discretisation option **reorder** can be used to trigger the renumbering of the mesh nodes, elements and faces using a Morton or Hilbert scheme.

More information on Morton and Hilbert curves can be found there for example:
https://en.wikipedia.org/wiki/Z-order_curve

This reordering improves the data locality in memory and thus allows a better cache usage. 

**IMPORTANT**: just renumbering one type of entity (say elements) is unlikely to trigger any benefit. The key idea is that when performing an indirection from face to element for example, the two returned elements should be always close in memory. If **both** elements and faces are renumbered so that their center of mass follows the same Z-curve (Hilbert or Morton), this is more likely to be the case.

In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.6_beta")

In [ ]:
from trustutils import run

run.reset()

discrets = ["VDF", "VEF"]
dims = [2,3]
algos = ["Morton", "Hilbert"]
z_cond = " 0. <= Z <= 1.0 "
bound_z =  r"""bord front Z = 0.0 0.  <= X <= 1  0. <= Y <= 1.0 
               bord back Z = 1.0 0.   <= X <= 1  0. <= Y <= 1.0"""
bc_z = r"""front paroi_fixe 
           back symetrie"""

def build_disc(disc, dim, algo):
    if disc == "VDF":
        return f"""
VDF dis 
read dis {{ 
    reorder {{  algo {algo} 
                dump }}
}}
"""
    else:
        tri_tet = "trianguler" if dim == 2 else "tetraedriser"
        vef_dis = f"""
{tri_tet} dom 
# VerifierCoin dom {{  }} #
VEFPreP1b dis 
read dis {{ P0 P1 changement_de_base_P1bulle 1 
   reorder {{ algo {algo}
             dump }}
}}"""
        return vef_dis

for dim in dims:
    for ze_disc in discrets:
        for algo in algos:
            sub_dir = f"upwind_{ze_disc}_{dim}D_{algo}"
            if dim == 2:
                nx, ny = 9,9
                params = {"dim": dim, "OZ": "", "NX": nx, "NY": ny, "NZ": "", "LZ": "",
                          "ZCOND": "", "BOUND_Z": "", "DISC": build_disc(ze_disc, dim, algo), "BC_Z": ""}
            else: # dim = 3
                nx, ny, nz = 5,5,5   # Not too much, otherwise we don't see anything on the plots!
                params = {"dim": dim, "OZ": "0.", "NX": nx, "NY": ny, "NZ": nz,  "LZ": "1.0",
                          "ZCOND": z_cond, "BOUND_Z": bound_z, "DISC": build_disc(ze_disc, dim, algo), "BC_Z": bc_z}
            run.addCaseFromTemplate("upwind_tpl.data", targetDirectory=sub_dir, dic=params)
        
run.printCases()
run.runCases()

The small utility script that will allow to plot the entity location (thanks ChatGPT!):

In [ ]:
import matplotlib.pyplot as plt

def read_points_from_file(filename, skip_first=0, last_idx=-1):
    points = []
    dim = -1
    try:
        with open(filename, 'r') as file:
            for line in file:
                # Split on whitespace and convert to float
                parts = line.strip().split()
                if len(parts) == 2:
                    dim = 2
                    x, y = map(float, parts)
                    points.append((x, y))
                elif len(parts) == 3:
                    dim = 3
                    x, y, z = map(float, parts)
                    points.append((x, y, z))
        if last_idx == -1:
            return points[skip_first:], dim
        else:
            return points[skip_first:last_idx], dim
    except FileNotFoundError as e:
        print(f"File {filename} not found!")
        return None, -1

def plot_points_with_gradient(points1, points2, title1="", title2=""):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, points, title in zip(axes, [points1, points2], [title1, title2]):
        x_vals, y_vals = zip(*points)

        norm = plt.Normalize(0, len(points) - 1)
        cmap = plt.get_cmap("viridis")

        for i in range(1, len(points)):
            ax.plot(x_vals[i-1:i+1], y_vals[i-1:i+1],
                    color=cmap(norm(i)), lw=2)

        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_title(title)
        ax.grid(True)
        ax.axis('equal')

    plt.tight_layout()
    plt.show()

def plot_points_3d_with_gradient(points1, points2, title1="", title2=""):
    fig = plt.figure(figsize=(12, 5))

    datasets = [(points1, title1), (points2, title2)]

    for idx, (points, title) in enumerate(datasets, start=1):
        ax = fig.add_subplot(1, 2, idx, projection='3d')

        x_vals, y_vals, z_vals = zip(*points)

        norm = plt.Normalize(0, len(points) - 1)
        cmap = plt.get_cmap("viridis")

        for i in range(1, len(points)):
            ax.plot([x_vals[i-1], x_vals[i]],
                    [y_vals[i-1], y_vals[i]],
                    [z_vals[i-1], z_vals[i]],
                    color=cmap(norm(i)), lw=2)

        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_title(title)

    plt.tight_layout()
    plt.show()

def visu_helper(disc, dim, algo, entity):
    points_bef, _ = read_points_from_file(run.BUILD_DIRECTORY + f"/upwind_{disc}_{dim}D_{algo}/reordering_{entity}_before.txt")
    points_aft, _ = read_points_from_file(run.BUILD_DIRECTORY + f"/upwind_{disc}_{dim}D_{algo}/reordering_{entity}_after.txt")
    if points_bef is None or points_aft is None: 
        return
    if dim == 2:
        plot_points_with_gradient(points_bef, points_aft, title1=f"Original {entity}", title2=f"Renumbered {entity} - {algo}")
    else:
        plot_points_3d_with_gradient(points_bef, points_aft, title1=f"Original {entity}", title2=f"Renumbered {entity} - {algo}")
    

## Reordering visualisation

Below we show on small cases the numbering scheme before and after the reordering for both available schemes : Morton and Hilbert.


### VEF - 2D results

The mesh:

In [ ]:
from trustutils import visit
visit.showMesh("upwind_VEF_2D_Morton/upwind_tpl.lata","dom")

#### Nodes
For nodes, before and after re-ordering:

In [ ]:
visu_helper("VEF", 2, "Morton", "som")

In [ ]:
visu_helper("VEF", 2, "Hilbert", "som")

#### Elements
For elements, before and after re-ordering:

In [ ]:
visu_helper("VEF", 2, "Morton", "elem")

In [ ]:
visu_helper("VEF", 2, "Hilbert", "elem")

#### Faces
Finally for faces, before and after re-ordering. Here we note : a. the non-standard faces are not considered (i.e. all the faces touching a boundary element), and also b. the reordering is not so impressive since the node and element reordering has already constrained the geometry significantly :

In [ ]:
visu_helper("VEF", 2, "Morton", "faces")

In [ ]:
visu_helper("VEF", 2, "Hilbert", "faces")

### VEF - 3D results
In 3D we just show the node renumbering :

For nodes, before and after re-ordering:

In [ ]:
visu_helper("VEF", 3, "Morton", "som")

In [ ]:
visu_helper("VEF", 3, "Hilbert", "som")

### VDF - 2D results

The mesh :

In [ ]:
from trustutils import visit
visit.showMesh("upwind_VDF_2D_Morton/upwind_tpl.lata","dom")

#### Nodes
For nodes, before and after re-ordering:

In [ ]:
visu_helper("VDF", 2, "Morton", "som")

In [ ]:
visu_helper("VDF", 2, "Hilbert", "som")

#### Elements
For elements, before and after re-ordering:

In [ ]:
visu_helper("VDF", 2, "Morton", "elem")

In [ ]:
visu_helper("VDF", 2, "Hilbert", "elem")

#### Faces
Finally for faces, before and after re-ordering - here we note the reordering is not so impressive since the node and element reordering has already constrained the geometry significantly :

In [ ]:
visu_helper("VDF", 2, "Morton", "faces")

In [ ]:
visu_helper("VDF", 2, "Hilbert", "faces")

### VDF - 3D results
In 3D we just show the node renumbering :

For nodes, before and after re-ordering:

In [ ]:
visu_helper("VDF", 3, "Morton", "som")

In [ ]:
visu_helper("VDF", 3, "Hilbert", "som")